In [1]:
using Pkg
Pkg.instantiate()
Pkg.update()

    Updating registry at `~/.julia/registries/General.toml`
    Updating git-repo `https://github.com/euriqa-brassboard/MSSim.jl.git`
     Project No packages added to or removed from `~/projects/yyc-data/euriqa/calculations/rydberg_czs/Project.toml`
    Manifest No packages added to or removed from `~/projects/yyc-data/euriqa/calculations/rydberg_czs/Manifest.toml`
        Info We haven't cleaned this depot up for a bit, running Pkg.gc()...
      Active manifest files: 9 found
      Active artifact files: 1 found
      Active scratchspaces: 0 found
     Deleted no artifacts, repos, packages or scratchspaces


In [2]:
include("sqrt_cz.jl")

opt_n! (generic function with 1 method)

In [3]:
using NPZ

In [4]:
Ω = 2π * 3
nseg = 30
nsubsample = 30
t_gate = 0.7
opt = SplineMultiOpt(Ω=Ω, nseg=nseg, nsubsample=nsubsample, t_gate=t_gate,
                     δωs=(-0.2, -0.1, 0.0, 0.1, 0.2),
                     weights=(0.05, 0.2, 1.0, 0.2, 0.05),
                     lam_robs=(0.0, 0.0001, 0.001, 0.0001, 0.0),
                     lam_leaks=(0.05, 0.2, 1.0, 0.2, 0.05),
                     lam_darks=(0.05, 0.2, 1.0, 0.2, 0.05));
# opt = Opt(Ω=Ω, num_slices=num_slices, t_gate=t_gate,
#           lam_rob=0.1, lam_leak=1, lam_dark=1);
# optional keyword arguments:
# algorithm=:LD_CCSAQ, maxeval_pre=1000, maxtime=3, xtol=1e-7, minω=-2π * 10, maxω=2π * 10

In [5]:
best_obj, best_args = @time opt_n!(opt, 100; verbose=false, pre_threshold=0.01) # default verbosity is true

obj = 0.01866368610129377
obj = 0.0004887126580264642
obj = 9.679812699713123e-5
Round 20 done
Round 40 done
Round 60 done
Round 80 done
Round 100 done
252.329340 seconds (2.56 M allocations: 127.909 MiB, 0.02% gc time, 0.72% compilation time)


(9.679812699713123e-5, [-62.83185307179586, -62.83185307179586, -62.83185307179586, -47.57796143933808, -8.833567123371243, -7.087606924324767, 24.469236233801734, 3.2916647018641525, 25.17922570625528, 7.328401624108252  …  46.63153215489173, -3.267221247397817, -15.677481016093413, 28.624369152849496, -12.941312565188992, -50.78377566768497, -62.83185307179586, -62.83185307179586, -62.83185307179586, -11.474237432579095])

In [6]:
# More tries to refine the result.
for _ in 1:25
    best_obj, best_args = @time opt_n!(opt, 40; pre_threshold=0.01,
                                       verbose=false, best_obj=best_obj, best_args=best_args)
    if best_obj < 0.0001
        break
    end
end

Round 20 done
obj = 9.428323352760706e-5
Round 40 done
110.380168 seconds (3.85 k allocations: 186.297 KiB, 0.01% compilation time)


In [7]:
best_ϕs = fm_to_phase(opt, best_args)
println(best_ϕs)

[0.0, -0.0330280411470443, -0.06626576777241489, -0.09970937659795598, -0.13335126106735595, -0.16718001134614732, -0.20118041432170716, -0.23533345360325636, -0.26961630952186, -0.3040023591304278, -0.33846117620371313, -0.37295853123831396, -0.4074563914526723, -0.4419129207870745, -0.4762824799036507, -0.5105156261863757, -0.5445591137410686, -0.5783558933953922, -0.6118451126988541, -0.6449621159228053, -0.6776384440604416, -0.7098018348268034, -0.7413762226587738, -0.7722817387150819, -0.8024347108762999, -0.8317476637448441, -0.8601293186449764, -0.8874845936228011, -0.9137146034462673, -0.9387166596051691, -0.9623842703111443, -0.9846071404976746, -1.0052732215497908, -1.0242769102228941, -1.0415210983724599, -1.0569171729540365, -1.07038501602325, -1.081853004735798, -1.0912580113474537, -1.0985454032140654, -1.1036690427915554, -1.1065912876359205, -1.1072829904032326, -1.1057234988496363, -1.101900655831353, -1.095810799304677, -1.0874587623259786, -1.0768578730517022, -1.064

In [8]:
npzwrite("sqrtcz_0.6us_3MHz.npz", Dict("phase_list"=>best_ϕs, "t_gate"=>t_gate, "Omega"=>Ω))